### Scrape Jobs

Python Job Scraping library "JobSpy": https://github.com/speedyapply/JobSpy

In [ ]:
from pathlib import Path
from jobspy import scrape_jobs
import pandas as pd
from IPython.display import HTML, display

In [ ]:
jobs = scrape_jobs(
    site_name=["linkedin", "indeed"],  # zip_recruiter, glassdoor, google, bayt, bdjobs
    linkedin_fetch_description=True,
    search_term='"AI Engineer"',
    location="USA",
    country_indeed="USA",
    job_type="fulltime",
    hours_old=720,  # 30 days
    results_wanted=50,
)

print(f"Total jobs scraped: {len(jobs)}")

In [ ]:
jobs_df = pd.DataFrame(jobs)

### Filter out jobs that don't posses the required attributes

In [ ]:
# We keep only jobs that have the fields we want to store later.
jobs_before_required_filter = len(jobs_df)

required_columns = ["title", "job_url", "description"]
has_required_values = (
    jobs_df[required_columns]
    .fillna("")
    .apply(lambda column: column.astype(str).str.strip() != "")
    .all(axis=1)
)
jobs_df = jobs_df[has_required_values].copy()

print(f"Jobs with title, job URL, and description: {len(jobs_df)}")
print(
    f"Jobs filtered out because of missing required fields: {jobs_before_required_filter - len(jobs_df)}"
)

### Filter out jobs that don't contain "AI" and "Engineer" in the title

In [ ]:
# We ensure the title contains BOTH "AI" and "Engineer".
# We need to do this because, for Indeed, the job descriptions are also searched.
# That means we could get jobs that only have "AI Engineer" in the description.
jobs_before_title_filter = len(jobs_df)

title_contains_ai = jobs_df["title"].str.contains("AI", case=False, na=False)
title_contains_engineer = jobs_df["title"].str.contains(
    "Engineer", case=False, na=False
)
mask = title_contains_ai & title_contains_engineer
matching_jobs = jobs_df[mask].copy()

print(f"Jobs matching 'AI' + 'Engineer' in title: {len(matching_jobs)}")
print(
    f"Jobs filtered out because of the title filter: {jobs_before_title_filter - len(matching_jobs)}"
)

### Filter out duplicates

In [ ]:
# We drop duplicates based on the combination of title and company.
jobs_before_deduplication = len(matching_jobs)

filtered_jobs = matching_jobs.drop_duplicates(subset=["title", "company"]).copy()

print(f"Jobs after deduplicating identical title/company pairs: {len(filtered_jobs)}")
print(
    f"Jobs filtered out as duplicates: {jobs_before_deduplication - len(filtered_jobs)}"
)

### Display jobs

In [ ]:
results_to_show = filtered_jobs[["title", "company", "job_url"]].copy()
results_to_show["job_url"] = results_to_show["job_url"].apply(
    lambda url: f'<a href="{url}" target="_blank">{url}</a>' if pd.notna(url) else ""
)

display(HTML(results_to_show.to_html(escape=False, index=False)))

### Save the jobs as JSONL file

In [ ]:
# Save the filtered jobs to a JSONL file in the "jobs" directory
# https://jsonlines.org/
output_dir = Path("jobs")
jsonl_path = output_dir / "1-scraped_jobs.jsonl"
jobs_to_save = filtered_jobs[["title", "job_url", "description"]].copy()
jobs_to_save.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)